# React agent 短期记忆

In [ ]:
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph,START,END,MessagesState
from langgraph.prebuilt import ToolNode
from model_factory import MiniMax_Model

# 使用内存来保存记忆
memory = MemorySaver()

@tool
def search(query: str):
    '''
    模拟访问网络查询天气
    :param query:
    :return:
    '''
    return "北京天气晴朗,气温22度,湿度30%"

tools = [search]
tool_node = ToolNode(tools)

model = MiniMax_Model.bind_tools(tools)

def should_continue(state:MessagesState):
    '''
    返回下一个需要执行的节点
    :param state:
    :return:
    '''
    last_message = state["messages"][-1]
    # 如果没有函数调用,则结束
    if not last_message.tool_calls:
        return END

    return "action"

# 定义调用模型的函数
def call_model(state:MessagesState):
    response = model.invoke( state["messages"])
    return {
        "messages":response,
    }

workflow = StateGraph(MessagesState)
workflow.add_node("agent",call_model)
workflow.add_node("action",tool_node)

workflow.add_edge(START,"agent")

workflow.add_conditional_edges(
    # 定义起始节点
    "agent",
    # 传入确定下一个调用节点的函数
    should_continue,
    # 传入路径映射
    {
        "action":"action",
        END:END
    }
)

workflow.add_edge("action","agent")

# 设置记忆点
app = workflow.compile(checkpointer=memory)


In [ ]:
from langchain.messages import HumanMessage

config = {
    "configurable":{
        "thread_id":"1"
    }
}

input_message = {
    "messages":[
        {"role":"user","content":"i am tomie"}
    ]
}
res = app.stream(input=input_message,config=config,stream_mode="values")
for chunk in res:
    chunk["messages"][-1].pretty_print()

config = {
    "configurable":{
        "thread_id":"1"
    }
}

input_message = {
    "messages":[
        {"role":"user","content":"what is my name"}
    ]
}
res = app.stream(input=input_message,config=config,stream_mode="values")
for chunk in res:
    chunk["messages"][-1].pretty_print()